In [ ]:
import pandas as pd
import os
from pathlib import Path
import requests
import csv
import numpy as np
from datetime import datetime
import time

Холодные кошельки

cold_wallets={'CKy3KzEMSL1PQV6Wppggoqi2nGA7teE4L7JipEK89yqj',
'G6zmnfSdG6QJaDWYwbGQ4dpCSUC4gvjfZxYQ4ZharV7C',
'85cPov8nuRCkJ88VNMcHaHZ26Ux85PbSrHW4jg7izW4h',
'VTvk7sG6QQ28iK3NEKRRD9fvPzk5pKpJL2iwgVqMFcL',
'D6gCBB3CZEMNbX1PDr3GtZAMhnebEumcgJ2yv8Etv5hF',
'146yGthSmnTPuCo6Zfbmr56YbAyWZ3rzAhRcT7tTF5ha',
'GXTrXayxMJUujsRTxYjAbkdbNvs6u2KN89UpG8f6eMAg',
'AzAvbCQsXurd2PbGLYcB61tyvE8kLDaZShE1S5Bp3WeS',
'3qP77PzrHxSrW1S8dH4Ss1dmpJDHpC6ATVgwy5FmXDEf',
'2PoMLnppbphUWkNReF15BJ4a6c4B348GnStSEv9tFsY1',
'E2RvJg2myWpKcbkhBuF81gfhYr6KvmNcDbSmr5qnatYy',
'9cNE6KBg2Xmf34FPMMvzDF8yUHMrgLRzBV3vD7b1JnUS',
'F7RkX6Y1qTfBqoX5oHoZEgrG1Dpy55UZ3GfWwPbM58nQ',
'CBEADkb8TZAXHjVE3zwad4L995GZE7rJcacJ7asebkVG',
'Cw32Ny2xcYdpfJmvFd7h2pRhcbcjoJFq8KrrA8YLwZeh',
'FyJBKcfcEBzGN74uNxZ95GxnCxeuJJujQCELpPv14ZfN',
'bazxfWH6Vu2FncaSKc3kkgvyCuPwyFpx4ryZPcNiG7x',
'42brAgAVNzMBP7aaktPvAmBSPEkehnFQejiZc53EpJFd',
'DLY8RdtrGELZh7Dnmf1tm99sfcXh6KdMtgjgRD1m1uQ7',
'DTNnXBh7JcKTzfWbqcuNYaYBQXS2fWWoXEdJ5iyNwvFX',
'7TWnq4WeYcwQWBCwKeEX2Q9xqVtthPGkB7adNvueuVuh',
'4S8C1yrRZmJYPzCqzEVjZYf6qCYWFoF7hWLRzssTCotX',
'G1d69HFvNiJc4gAJAtjv4DkePPAYv6Z3vttY5XCTq3vN'
}

Формирование единого датасета из собранных данных

In [2]:
ROOT = Path(r"C:\Users\User\my\py\practise\solana\downloads1")   # папка с подпапками
OUT  = r"C:\Users\User\my\py\practise\solana\combined_from_01_2020.csv"

files = list(ROOT.rglob("*.csv"))
print("Найдено файлов:", len(files))

first_file = True
with open(OUT, "w", newline="", encoding="utf-8") as fout:
    writer = None

    for path in files:
        with open(path, "r", encoding="utf-8") as fin:
            reader = csv.reader(fin)
            header = next(reader)

            if first_file:
                writer = csv.writer(fout)
                writer.writerow(header)
                first_file = False

            writer.writerows(reader)

print("Готово:", OUT)

Найдено файлов: 41465
Готово: C:\Users\User\my\py\practise\solana\combined_from_01_2020.csv


Округляем вниз, 23.57=23.00

Время плюс три часа, UTC

In [3]:
df_final=pd.read_csv(r"C:\Users\User\my\py\practise\solana\combined_from_01_2020.csv")
df_final= df_final.drop_duplicates().reset_index(drop=True)
df_final["Human Time"] = pd.to_datetime(df_final["Human Time"], utc=True, errors="coerce")
df_final=df_final.sort_values(by='Human Time').reset_index(drop=True)
df_final["date"] = df_final["Human Time"].dt.date
df_final['hour'] = df_final['Human Time'].dt.floor('H')
df_final["minute"] = df_final["Human Time"].dt.floor("T")          # 1 минута
df_final["minute_5"] = df_final["Human Time"].dt.floor("5T")       # 5 минут
df_final["minute_15"] = df_final["Human Time"].dt.floor("15T")
df_final['Amount']=df_final['Amount']/1000000000
df_final = df_final[
    ['Human Time','date', 'hour','minute','minute_5','minute_15','Amount', 'Flow', 'Value', 'From', 'To']
]

C:\Users\User\AppData\Local\Temp\ipykernel_3984\1115747526.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_final['hour'] = df_final['Human Time'].dt.floor('H')
C:\Users\User\AppData\Local\Temp\ipykernel_3984\1115747526.py:7: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_final["minute"] = df_final["Human Time"].dt.floor("T")          # 1 минута
C:\Users\User\AppData\Local\Temp\ipykernel_3984\1115747526.py:8: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_final["minute_5"] = df_final["Human Time"].dt.floor("5T")       # 5 минут
C:\Users\User\AppData\Local\Temp\ipykernel_3984\1115747526.py:9: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_final["minute_15"] = df_final["Human Time"].dt.floor("15T")


In [ ]:
df_final 

,Human Time,date,hour,minute,minute_5,minute_15,Amount,Flow,Value,From,To
0,2020-10-02 21:39:59+00:00,2020-10-02,2020-10-02 21:00:00+00:00,2020-10-02 21:39:00+00:00,2020-10-02 21:35:00+00:00,2020-10-02 21:30:00+00:00,7.6800,out,-1.0,ASTyfSima4LLAdDgoFGkgqoKowG1LZFDr9fAQrg7iaJZ,26RcRicG13BFJ8WKJvpwH7MQB8MDp5hSs1egdazxzwjk
1,2020-10-03 00:41:52+00:00,2020-10-03,2020-10-03 00:00:00+00:00,2020-10-03 00:41:00+00:00,2020-10-03 00:40:00+00:00,2020-10-03 00:30:00+00:00,5.5340,out,-1.0,5VCwKtCXgCJ6kit5FybXjvriW3xELsFDhYrPSqtJNmcD,CjY7kHhfVPzoJ3RmsqL3fm5jD6ncPRbY1vQB6UYjoajU
2,2020-10-03 00:48:26+00:00,2020-10-03,2020-10-03 00:00:00+00:00,2020-10-03 00:48:00+00:00,2020-10-03 00:45:00+00:00,2020-10-03 00:45:00+00:00,5.5260,out,-1.0,5VCwKtCXgCJ6kit5FybXjvriW3xELsFDhYrPSqtJNmcD,CjY7kHhfVPzoJ3RmsqL3fm5jD6ncPRbY1vQB6UYjoajU
3,2020-10-03 00:53:34+00:00,2020-10-03,2020-10-03 00:00:00+00:00,2020-10-03 00:53:00+00:00,2020-10-03 00:50:00+00:00,2020-10-03 00:45:00+00:00,5.5390,out,-1.0,5VCwKtCXgCJ6kit5FybXjvriW3xELsFDhYrPSqtJNmcD,CjY7kHhfVPzoJ3RmsqL3fm5jD6ncPRbY1vQB6UYjoajU
4,2020-10-03 01:02:05+00:00,2020-10-03,2020-10-03 01:00:00+00:00,2020-10-03 01:02:00+00:00,2020-10-03 01:00:00+00:00,2020-10-03 01:00:00+00:00,5.4640,out,-1.0,5VCwKtCXgCJ6kit5FybXjvriW3xELsFDhYrPSqtJNmcD,CjY7kHhfVPzoJ3RmsqL3fm5jD6ncPRbY1vQB6UYjoajU
...,...,...,...,...,...,...,...,...,...,...,...
36032546,NaT,NaT,NaT,NaT,NaT,NaT,50.0000,out,-1.0,FxteHmLwG9nk1eL4pjNve3Eub2goGkkz6g6TbvdmW46a,CnLhmg91Go9YctBQKG1uXCf4Z1EhfTdbJ34wq1H16Rx6
36032547,NaT,NaT,NaT,NaT,NaT,NaT,111.0000,out,-1.0,FxteHmLwG9nk1eL4pjNve3Eub2goGkkz6g6TbvdmW46a,Er62214s6ziA1o9BMB7rYNntVz2sNw2tjStBrZxqdGFa
36032548,NaT,NaT,NaT,NaT,NaT,NaT,12390.0000,out,-1.0,FxteHmLwG9nk1eL4pjNve3Eub2goGkkz6g6TbvdmW46a,28vzLbZ4pNXRCh9xwoSD4PHhCDUdXBVaCaid4Czjvvwd
36032549,NaT,NaT,NaT,NaT,NaT,NaT,3.2245,out,-1.0,u6PJ8DtQuPFnfmwHbGFULQ4u4EgjDiyYKjVEsynXq2w,6eu9vV2SpJhSffuT5vXsg7VCi4vCBUTX8nx8WrDJN8vd


In [14]:
df_final.columns

Index(['Signature', 'Block Time', 'Human Time', 'Action', 'From', 'To',
       'Amount', 'Flow', 'Value', 'Decimals', 'Token Address', ' Multiplier',
       'date', 'hour'],
      dtype='object')

Функции для формирования очейн фичей

In [ ]:
def shannon_entropy(series: pd.Series) -> float:
    """Шэнноновская энтропия распределения (адресов и т.п.)."""
    if series.empty:
        return 0.0
    probs = series.value_counts(normalize=True)
    return float(-(probs * np.log(probs + 1e-12)).sum())


def burstiness_cv(ts: pd.Series) -> float:
    """
    Коэффициент вариации интервалов между событиями:
    std(Δt) / mean(Δt), где Δt — интервалы в секундах.
    """
    if ts.size < 3:
        return 0.0
    s = ts.sort_values()
    dt = s.diff().dt.total_seconds().dropna()
    if dt.size < 2:
        return 0.0
    m = dt.mean()
    if m <= 0:
        return 0.0
    return float(dt.std() / (m + 1e-9))


def prepare_base(df: pd.DataFrame) -> pd.DataFrame:
    """Приводим типы и создаём вспомогательные столбцы."""
    df = df.copy()

    # timestamp для burstiness и агрегаций
    df["Human Time"] = pd.to_datetime(df["Human Time"], utc=True, errors="coerce")
    df["ts"] = df["Human Time"]

    # метки in/out
    flow = df["Flow"].astype(str).str.lower()
    df["is_in"] = flow.eq("in")
    df["is_out"] = flow.eq("out")

    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce").fillna(0.0)

    df["amount_in"] = np.where(df["is_in"], df["Amount"], 0.0)
    df["amount_out"] = np.where(df["is_out"], df["Amount"], 0.0)
    df["ts"] = pd.to_datetime(df["Human Time"], utc=True, errors="coerce")
    q = df["ts"].dt.to_period("Q")

# 95-й перцентиль Amount внутри каждого квартала
    df["whale_threshold_q"] = df.groupby(q)["Amount"].transform(lambda s: s.quantile(0.95))
    df["is_whale"] = df["Amount"] > df["whale_threshold_q"]
    df["whale_amount"] = np.where(df["is_whale"], df["Amount"], 0.0)

    return df


# ================= ОСНОВНАЯ ФУНКЦИЯ АГРЕГАЦИИ =================

def build_features_table(df: pd.DataFrame, level: str = "hourly") -> pd.DataFrame:
    """
    Строит таблицу фич для нужного таймфрейма.
    level:
        "1min"   — 1 минута
        "5min"   — 5 минут
        "15min"  — 15 минут
        "hourly" — 1 час
        "daily"  — 1 день
    """

    # ---- настройки под каждый таймфрейм ----
    # freq    — как floor'ить время
    # minutes — длительность окна в минутах (для velocity_tpm)
    # win_short / win_long_tx — окна для rolling (в количестве периодов)
    # win_spike — окно для spike-индикаторов (в количестве периодов)
    config = {
        "1min":   {"freq": "T",   "minutes": 1,   "win_short": 15, "win_long_tx": 60,  "win_spike": 60},
        "5min":   {"freq": "5T",  "minutes": 5,   "win_short": 3,  "win_long_tx": 12,  "win_spike": 12},
        "15min":  {"freq": "15T", "minutes": 15,  "win_short": 4,  "win_long_tx": 96,  "win_spike": 96},
        "hourly": {"freq": "H",   "minutes": 60,  "win_short": 3,  "win_long_tx": 24,  "win_spike": 24},
        "daily":  {"freq": "D",   "minutes": 1440,"win_short": 3,  "win_long_tx": 24,  "win_spike": 7},
    }

    if level not in config:
        raise ValueError(f"Unsupported level='{level}'. Use one of {list(config.keys())}")

    cfg = config[level]

    df = prepare_base(df)

    # ---- строим временной "бакет" нужного размера ----
    df = df.copy()
    df["bucket"] = df["Human Time"].dt.floor(cfg["freq"])

    # ---- группировка по этому бакету ----
    g = df.groupby("bucket", sort=True)

    # ===== базовые агрегаты =====
    agg = pd.DataFrame(index=g.size().index)

    # tx_count: количество транзакций (или уникальных сигнатур)
    if "Signature" in df.columns:
        agg["tx_count"] = g["Signature"].nunique()
    else:
        agg["tx_count"] = g.size()

    # address_count, receiver_count, unique_addresses
    agg["address_count"] = g["From"].nunique()
    agg["receiver_count"] = g["To"].nunique()
    agg["unique_addresses"] = g[["From", "To"]].apply(
        lambda x: pd.concat([x["From"], x["To"]]).nunique()
    )

    # inflow, outflow, total_volume
    agg["inflow"] = g["amount_in"].sum()
    agg["outflow"] = g["amount_out"].sum()
    agg["total_volume"] = agg["inflow"] + agg["outflow"]

    # net_flow
    agg["net_flow"] = agg["inflow"] - agg["outflow"]

    # mean/std/median/max/min по Amount
    agg["mean_amount"] = g["Amount"].mean()
    agg["std_amount"] = g["Amount"].std()
    agg["median_amount"] = g["Amount"].median()
    agg["max_amount"] = g["Amount"].max()
    agg["min_amount"] = g["Amount"].min()

    # китовые фичи
    agg["whale_tx_count"] = g["is_whale"].sum()
    agg["whale_volume"] = g["whale_amount"].sum()

    # address_entropy & burstiness
    agg["address_entropy"] = g["From"].apply(shannon_entropy)
    agg["burstiness"] = g["ts"].apply(burstiness_cv)

    # ===== производные фичи (отношения, скорости, волатильности) =====

    # inflow_rel, outflow_rel, total_volume_rel (делим на net_flow)
    net = agg["net_flow"].replace(0, np.nan)
    agg["inflow_rel"] = agg["inflow"] / net
    agg["outflow_rel"] = agg["outflow"] / net
    agg["total_volume_rel"] = agg["total_volume"] / net

    # inflow_ratio, outflow_ratio (делим на total_volume)
    tv = agg["total_volume"].replace(0, np.nan)
    agg["inflow_ratio"] = agg["inflow"] / tv
    agg["outflow_ratio"] = agg["outflow"] / tv

    # avg_tx_amount
    agg["avg_tx_amount"] = agg["total_volume"] / agg["tx_count"].replace(0, np.nan)

    # value_volatility = std_amount / mean_amount
    agg["value_volatility"] = agg["std_amount"] / agg["mean_amount"].replace(0, np.nan)

    # whale_volume_share, avg_whale_amount
    agg["whale_volume_share"] = agg["whale_volume"] / tv
    agg["avg_whale_amount"] = agg["whale_volume"] / agg["whale_tx_count"].replace(0, np.nan)

    # скорость сети: транзакций в минуту
    agg["velocity_tpm"] = agg["tx_count"] / cfg["minutes"]

    # ===== rolling-фичи (на агрегированном ряду) =====
    agg = agg.reset_index().rename(columns={"bucket": "timestamp"})
    agg = agg.sort_values("timestamp").set_index("timestamp")

    win_short = cfg["win_short"]
    win_long_tx = cfg["win_long_tx"]
    win_spike = cfg["win_spike"]

    # rolling_netflow_3h / 24h (в терминах "короткого" и "длинного" окна)
    agg["rolling_netflow_3h"] = agg["net_flow"].rolling(win_short, min_periods=1).sum()
    agg["rolling_netflow_24h"] = agg["net_flow"].rolling(win_long_tx, min_periods=1).sum()

    # rolling_volume_3h / 24h
    agg["rolling_volume_3h"] = agg["total_volume"].rolling(win_short, min_periods=1).sum()
    agg["rolling_volume_24h"] = agg["total_volume"].rolling(win_long_tx, min_periods=1).sum()

    # нормированные rolling netflow
    rv3 = agg["rolling_volume_3h"].replace(0, np.nan)
    rv24 = agg["rolling_volume_24h"].replace(0, np.nan)
    agg["rolling_netflow_3h_normed"] = agg["rolling_netflow_3h"] / rv3
    agg["rolling_netflow_24h_normed"] = agg["rolling_netflow_24h"] / rv24

    # ===== spike-индикаторы =====
    agg["rolling_tx_mean_spike"] = agg["tx_count"].rolling(win_spike, min_periods=1).mean()
    agg["rolling_vol_mean_spike"] = agg["total_volume"].rolling(win_spike, min_periods=1).mean()
    agg["rolling_netflow_mean_spike"] = agg["net_flow"].rolling(win_spike, min_periods=1).mean()
    agg["rolling_inflow_mean_spike"] = agg["inflow"].rolling(win_spike, min_periods=1).mean()
    agg["rolling_outflow_mean_spike"] = agg["outflow"].rolling(win_spike, min_periods=1).mean()
    agg["rolling_addresses_mean_spike"] = agg["unique_addresses"].rolling(win_spike, min_periods=1).mean()
    agg["rolling_whale_share_mean_spike"] = agg["whale_volume_share"].rolling(win_spike, min_periods=1).mean()

    agg["spike_tx"] = agg["tx_count"] / agg["rolling_tx_mean_spike"].replace(0, np.nan)
    agg["spike_volume"] = agg["total_volume"] / agg["rolling_vol_mean_spike"].replace(0, np.nan)
    agg["spike_netflow"] = agg["net_flow"] / agg["rolling_netflow_mean_spike"].replace(0, np.nan)
    agg["spike_inflow"] = agg["inflow"] / agg["rolling_inflow_mean_spike"].replace(0, np.nan)
    agg["spike_outflow"] = agg["outflow"] / agg["rolling_outflow_mean_spike"].replace(0, np.nan)
    agg["spike_unique_addresses"] = agg["unique_addresses"] / agg["rolling_addresses_mean_spike"].replace(0, np.nan)
    agg["spike_whales"] = agg["whale_volume_share"] / agg["rolling_whale_share_mean_spike"].replace(0, np.nan)

    # вернём обычный индекс, добавим дату/время отдельно (на всякий случай)
    agg = agg.reset_index()
    agg["date"] = agg["timestamp"].dt.date
    agg["time"] = agg["timestamp"].dt.time

    # порядок колонок: timestamp, date, time, затем фичи
    base_cols = ["timestamp", "date", "time"]
    other_cols = [c for c in agg.columns if c not in base_cols]
    agg = agg[base_cols + other_cols]

    return agg

Формирование датасета агрегированных ончейн- данных

In [ ]:
features_daily  = build_features_table(df_final, level="daily")
features_daily = features_daily[features_daily["timestamp"] >= pd.Timestamp("2020-01-01", tz='UTC')]

C:\Users\User\AppData\Local\Temp\ipykernel_3984\1074225445.py:44: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  q = df["ts"].dt.to_period("Q")
C:\Users\User\AppData\Local\Temp\ipykernel_3984\1074225445.py:89: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["bucket"] = df["Human Time"].dt.floor(cfg["freq"])
C:\Users\User\AppData\Local\Temp\ipykernel_3984\1074225445.py:44: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  q = df["ts"].dt.to_period("Q")


In [80]:
features_daily

,timestamp,date,tx_count,address_count,receiver_count,unique_addresses,inflow,outflow,total_volume,net_flow,...,rolling_outflow_mean_spike,rolling_addresses_mean_spike,rolling_whale_share_mean_spike,spike_tx,spike_volume,spike_netflow,spike_inflow,spike_outflow,spike_unique_addresses,spike_whales
303,2023-08-01 00:00:00+00:00,2023-08-01,6328,1745,2938,4385,1.368549e+06,1.673687e+06,3.042236e+06,-305137.878890,...,9.771706e+05,4111.285714,0.963923,1.070288,1.631642,3.397353,1.542281,1.712789,1.066576,1.013997
304,2023-08-02 00:00:00+00:00,2023-08-02,6085,1704,2850,4269,4.017268e+05,5.630391e+05,9.647660e+05,-161312.315061,...,9.464471e+05,4104.571429,0.959246,1.032856,0.550958,1.137408,0.499274,0.594898,1.040060,0.970318
305,2023-08-03 00:00:00+00:00,2023-08-03,6172,1600,3000,4346,9.152622e+05,8.830170e+05,1.798279e+06,32245.199202,...,9.647959e+05,4126.857143,0.959434,1.044761,1.018047,-0.197593,1.141786,0.915237,1.053102,1.004901
306,2023-08-04 00:00:00+00:00,2023-08-04,6094,1594,2947,4254,3.880074e+05,5.891876e+05,9.771950e+05,-201180.198318,...,8.904512e+05,4135.428571,0.955107,1.032956,0.599193,1.340742,0.524051,0.661673,1.028672,0.980959
307,2023-08-05 00:00:00+00:00,2023-08-05,5445,1448,2731,3922,7.406701e+05,4.808569e+05,1.221527e+06,259813.232728,...,8.802629e+05,4187.142857,0.957194,0.914533,0.723176,-3.638204,0.915707,0.546265,0.936677,0.996270
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1132,2025-11-07 00:00:00+00:00,2025-11-07,28605,7448,13734,20662,1.907430e+06,1.850332e+06,3.757763e+06,57097.773541,...,2.002107e+06,19312.714286,0.951843,1.077534,0.933266,2.566165,0.942240,0.924193,1.069865,0.992230
1133,2025-11-08 00:00:00+00:00,2025-11-08,22340,5648,10914,16226,5.791186e+05,5.602373e+05,1.139356e+06,18881.373195,...,1.788333e+06,19263.714286,0.937837,0.840698,0.317031,1.099964,0.320753,0.313274,0.842309,0.925831
1134,2025-11-09 00:00:00+00:00,2025-11-09,20924,5503,10284,15426,6.326771e+05,6.725319e+05,1.305209e+06,-39854.825969,...,1.570532e+06,19256.000000,0.926756,0.787890,0.413449,-2.519405,0.398825,0.428219,0.801101,0.963232
1135,2025-11-10 00:00:00+00:00,2025-11-10,25598,6436,12156,18192,1.363686e+06,1.215638e+06,2.579324e+06,148048.273282,...,1.457169e+06,18887.571429,0.924446,0.981992,0.877090,5.600405,0.919171,0.834246,0.963173,1.008098


In [ ]:
features_daily.to_csv(r'C:\Users\User\my\py\practise\solana\features\features_daily_2020.csv')